# Tidywells: explore the open dataset

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jay-Kimani/TidyWells/blob/main/examples/open_data_demo.ipynb)

No regulator files needed. This notebook loads the open monthly dataset from tidywells.com, plots it, looks at our live forecast log, and fits a decline curve. Informational only, not investment, legal or engineering advice.

In [ ]:
%pip install -q git+https://github.com/Jay-Kimani/TidyWells.git

## Load the open dataset
One row per province, product and month. Volumes are in the regulators' units (oil m³, gas 10³ m³); we convert to daily rates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://tidywells.com/data/tidywells-monthly-production.csv"
df = pd.read_csv(URL)
conv = {"OIL": 6.2898 / 1000, "GAS": 35.3147 / 1000}   # thousand bbl, MMcf
df["rate"] = df.volume / df.days_in_month * df["product"].map(conv)   # thousand bbl/d (oil), MMcf/d (gas)
df["month"] = pd.PeriodIndex(df["month"], freq="M")
df.tail()

## Year-on-year change by series

In [ ]:
latest = df.month.max()
wide = df.pivot_table(index="month", columns=["province", "product"], values="rate")
yoy = (wide.iloc[-1] / wide.shift(12).iloc[-1] - 1) * 100
print(f"Latest month: {latest}")
yoy.round(1).rename("YoY %")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
wide[("AB", "OIL")].plot(ax=ax)
ax.set_title("Alberta oil, thousand bbl/d")
ax.set_xlabel("")
plt.show()

## Our forecast scorecard
Forecasts are written down before the data exists and graded later. See https://tidywells.com/scorecard.html.

In [ ]:
import json, urllib.request
log = json.load(urllib.request.urlopen("https://tidywells.com/data/forecasts.json"))["entries"]
f = pd.DataFrame(log)
print(len(f), "forecasts logged,", f.actual.notna().sum(), "graded")
f[(f.method == "seasonal_trend") & (f.series == "AB-OIL")][["made", "target", "forecast", "actual", "ape_pct"]]

## Fit a decline curve
The same bounded Arps fit the browser tool uses. Here on a simulated well; with your own Petrinex or BCER files use `tw.read_volumetrics` or `tw.read_bc_production` and `tw.monthly_series`.

In [ ]:
import numpy as np
import tidywells as tw

t = np.arange(36) / 12                      # years
q = tw.arps_rate(t, 150, 1.2, 1.0) * np.exp(np.random.default_rng(0).normal(0, 0.05, 36))
fit = tw.fit_arps(t, q)
fit